In [2]:
import sqlite3
import pandas as pd
con=sqlite3.connect('../data/raw/ipl.db')

def q(sql):
    return pd.read_sql(sql, con)

In [3]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()
    con.executescript(script)
    con.commit()
    print('ran',path)

In [4]:
q("""
-- what is the strike rate of every batter in the death overs

SELECT 
    batter,
    ROUND(
        100.0 * SUM(batter_runs) / SUM(
            CASE
                WHEN is_wide_ball = 0 AND is_no_ball = 0 
                THEN 1
                ELSE 0
            END
        ),2
    ) AS strike_rate
FROM deliveries
WHERE over_number >= 15 AND is_super_over = 0
GROUP BY batter;""")

,batter,strike_rate
0,A Ashish Reddy,146.58
1,A Badoni,165.03
2,A Chandila,57.14
3,A Chopra,50.00
4,A Choudhary,125.00
...,...,...
637,Yash Dayal,50.00
638,Yashpal Singh,130.77
639,Yudhvir Singh,166.67
640,Yuvraj Singh,183.40


In [5]:
run_sql_file('../sql/09_analysis_views.sql')

ran ../sql/09_analysis_views.sql


In [6]:
q("""SELECT *
FROM v_ball;""")

,season_id,match_id,batter,bowler,non_striker,over_number,ball_number,batter_runs,extras,total_runs,...,is_super_over,innings,batting_team_id,bowling_team_id,batting_team,bowling_team,phase,is_legal,bowler_runs,bowler_wicket
0,2008,335982,SC Ganguly,P Kumar,BB McCullum,0,0,0,1,1,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
1,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,1,0,0,0,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
2,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,2,0,1,1,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,0,1,0
3,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,3,0,0,0,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
4,2008,335982,BB McCullum,P Kumar,SC Ganguly,0,4,0,0,0,...,0,1,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Powerplay,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288046,2026,1529286,T Stubbs,JC Archer,Ashutosh Sharma,18,3,1,0,1,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,1,0
288047,2026,1529286,Ashutosh Sharma,JC Archer,T Stubbs,18,4,1,0,1,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,1,0
288048,2026,1529286,T Stubbs,JC Archer,Ashutosh Sharma,18,5,4,0,4,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,4,0
288049,2026,1529286,Ashutosh Sharma,Brijesh Sharma,T Stubbs,19,0,4,0,4,...,0,2,252,134,Delhi Capitals,Rajasthan Royals,Death,1,4,0


In [7]:
q("""SELECT phase, COUNT(*) FROM v_ball GROUP BY phase;"""  )

,phase,COUNT(*)
0,Death,65488
1,Middle,132173
2,Powerplay,90390


In [8]:
q("""SELECT count(*) FROM v_match_totals;""")

,count(*)
0,1187


In [9]:
q("""SELECT count(*) FROM v_innings;""")

,count(*)
0,2418


In [10]:
q("""SELECT phase,count(*) FROM v_ball;""")

,phase,count(*)
0,Powerplay,288051


In [11]:
q("""
SELECT batter,sum (Batter_runs) as total_runs
FROM v_ball
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 5;
 """)

,batter,total_runs
0,V Kohli,9040
1,RG Sharma,7183
2,S Dhawan,6769
3,DA Warner,6565
4,KL Rahul,5655


In [12]:
q("""SELECT batter,
        sum(CASE WHEN batter_runs=6
            THEN 1
            ELSE 0
        END) AS Total_sixes
FROM v_ball
GROUP BY batter
ORDER BY Total_sixes DESC
LIMIT 10;""")

,batter,Total_sixes
0,CH Gayle,357
1,RG Sharma,311
2,V Kohli,307
3,MS Dhoni,264
4,AB de Villiers,251
5,DA Warner,236
6,SV Samson,234
7,KL Rahul,232
8,AD Russell,223
9,KA Pollard,223


In [13]:
q("""SELECT batter, 

       SUM(is_legal) AS balls,

       SUM(batter_runs) AS runs,

       ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS strike_rate

FROM v_ball

GROUP BY batter

ORDER BY balls DESC

LIMIT 10;""")

,batter,balls,runs,strike_rate
0,V Kohli,6730,9040,134.32
1,RG Sharma,5396,7183,133.12
2,S Dhawan,5302,6769,127.67
3,DA Warner,4675,6565,140.43
4,AM Rahane,4141,5194,125.43
5,KL Rahul,4059,5655,139.32
6,SK Raina,4022,5528,137.44
7,MS Dhoni,3935,5439,138.22
8,RV Uthappa,3782,4952,130.94
9,KD Karthik,3567,4842,135.74


In [14]:
q("""
--batter who scored the highest share of runs in boundary

select batter,
         sum(batter_runs) as runs,
         round(100.0 *sum(case when batter_runs in (4,6) 
         then batter_runs
         else 0
         end)/sum(batter_runs),2)as boundary_perc

from v_ball
group by batter
having runs>2000
order by boundary_perc desc
limit 10; 
""")

,batter,runs,boundary_perc
0,AD Russell,2651,78.54
1,CH Gayle,4965,75.77
2,Abhishek Sharma,2241,73.54
3,YBK Jaiswal,2478,73.28
4,AC Gilchrist,2069,72.89
5,V Sehwag,2728,72.29
6,N Pooran,2375,70.65
7,DR Smith,2385,70.52
8,SR Watson,3874,68.25
9,GJ Maxwell,2819,68.04


In [15]:
q("""SELECT batter,SUM(is_legal) AS balls,SUM(batter_runs) AS runs,
ROUND(100.0* SUM(batter_runs)/SUM(is_legal),2) AS run_rate

FROM v_ball
WHERE phase ='Middle'
GROUP BY batter
ORDER BY run_rate desc
LIMIT 20;
""")

,batter,balls,runs,run_rate
0,Auqib Nabi,1,4,400.00
1,Avesh Khan,4,12,300.00
2,Kamran Akmal,20,51,255.00
3,Akash Deep,2,5,250.00
4,KAJ Roach,2,5,250.00
5,Urvil Patel,11,26,236.36
6,V Suryavanshi,89,189,212.36
7,Priyansh Arya,110,225,204.55
8,Kartik Tyagi,2,4,200.00
9,C Connolly,90,178,197.78


In [16]:
q("""SELECT batter,SUM(is_legal) AS balls,SUM(batter_runs) AS runs,
ROUND(100.0* SUM(batter_runs)/SUM(is_legal),2) AS run_rate

FROM v_ball
WHERE phase ='Middle'
GROUP BY batter
HAVING balls> 500
ORDER BY run_rate desc
LIMIT 20;
""")

,batter,balls,runs,run_rate
0,V Sehwag,611,1062,173.81
1,RM Patidar,566,970,171.38
2,N Pooran,848,1351,159.32
3,H Klaasen,697,1099,157.68
4,CH Gayle,1285,2011,156.50
5,AD Russell,767,1182,154.11
6,SR Watson,1302,2006,154.07
7,GJ Maxwell,1163,1780,153.05
8,SE Marsh,971,1448,149.12
9,DR Smith,675,1004,148.74


In [17]:
q("""SELECT batter,SUM(is_legal) AS balls,SUM(batter_runs) AS balls,
ROUND(100.0* SUM(batter_runs)/SUM(is_legal),2) AS strike_rate

FROM v_ball
WHERE phase ='Death'
GROUP BY batter
HAVING balls> 50
ORDER BY strike_rate desc
LIMIT 10;
""")

,batter,balls,balls,strike_rate
0,HM Amla,51,122,239.22
1,R Shepherd,73,168,230.14
2,AB de Villiers,829,1868,225.33
3,LS Livingstone,156,345,221.15
4,B Sai Sudharsan,118,246,208.47
5,CH Gayle,280,581,207.50
6,T Stubbs,257,533,207.39
7,Q de Kock,116,240,206.90
8,MA Agarwal,122,252,206.56
9,RR Pant,513,1035,201.75


In [18]:
q("""SELECT batter,
    SUM(is_legal) AS balls,
    SUM(batter_runs) AS runs,
    ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS strike_rate
FROM v_ball
WHERE innings = 2
GROUP BY batter
HAVING balls > 500
ORDER BY strike_rate DESC
LIMIT 10;""")

,batter,balls,runs,strike_rate
0,SP Narine,507,869,171.40
1,AD Russell,660,1130,171.21
2,Abhishek Sharma,648,1108,170.99
3,N Pooran,729,1218,167.08
4,V Sehwag,869,1427,164.21
5,GJ Maxwell,734,1164,158.58
6,P Simran Singh,527,832,157.87
7,JC Buttler,1461,2289,156.67
8,YBK Jaiswal,875,1351,154.40
9,SA Yadav,1382,2131,154.20


In [19]:
q("""SELECT batter,
        SUM(is_legal) AS balls,
        SUM(CASE WHEN 
            batter_runs=0 AND is_legal=1
            THEN 1 ELSE 0
            END) AS dot_balls,
        ROUND(100.0*SUM(CASE WHEN 
            batter_runs=0 AND is_legal=1
            THEN 1 ELSE 0
            END)/SUM(is_legal),2) AS
            dot_pct

FROM v_ball
WHERE phase='Death'
GROUP BY batter
HAVING balls >=500
ORDER BY dot_pct ASC
LIMIT 5;
""")

,batter,balls,dot_balls,dot_pct
0,JP Duminy,516,104,20.16
1,AB de Villiers,829,175,21.11
2,SK Raina,521,111,21.31
3,KL Rahul,502,110,21.91
4,V Kohli,807,185,22.92


In [20]:
q("""
--how do runs and wickets differ across the three phases of an innings?

select phase,sum(is_legal)as balls,sum(total_runs),
round(6.0*sum(total_runs)/sum(is_legal),2)as runs_per_over,
round(100.0*sum(is_wicket)/sum(is_legal),2)as wicket_perc

from v_ball
group by phase;

""")

,phase,balls,sum(total_runs),runs_per_over,wicket_perc
0,Death,62205,104056,10.04,8.46
1,Middle,128191,169160,7.92,4.34
2,Powerplay,86921,116397,8.03,4.01


In [21]:
q("""
SELECT over_number,
       SUM(total_runs) AS total_runs
FROM v_ball
GROUP BY over_number
ORDER BY total_runs DESC
LIMIT 5;
""")

,over_number,total_runs
0,17,21722
1,18,21063
2,16,21001
3,5,20917
4,4,20862


In [22]:
q("""
SELECT over_number,
       SUM(total_runs) AS total_runs
FROM v_ball
GROUP BY over_number
ORDER BY total_runs DESC
LIMIT 5;
""")

,over_number,total_runs
0,17,21722
1,18,21063
2,16,21001
3,5,20917
4,4,20862


In [23]:
q("""
select over_number+1 as overs,
round (6.0*sum(total_runs)/sum(is_legal),2)as runs_per_over
from v_ball
group by over_number
order by runs_per_over desc;
""")


,overs,runs_per_over
0,20,11.39
1,19,10.53
2,18,10.12
3,17,9.46
4,16,9.05
5,15,8.74
6,6,8.68
7,5,8.65
8,4,8.63
9,14,8.43


In [24]:
q("""
SELECT 
    ROUND(AVG(powerplay_score), 2) AS average_powerplay_score
FROM (
    SELECT 
        match_id,
        innings,
        SUM(total_runs) AS powerplay_score
    FROM v_ball
    WHERE over_number BETWEEN 0 AND 5
    GROUP BY match_id, innings
);
""")

,average_powerplay_score
0,48.14


In [25]:
q("""
--which over sees the most wikets fall?

SELECT
    over_number + 1 AS over,
    SUM(is_wicket) AS wickets
FROM v_ball
GROUP BY over_number
ORDER BY over_number DESC;
""")

,over,wickets
0,20,1433
1,19,1089
2,18,1055
3,17,863
4,16,824
5,15,764
6,14,703
7,13,661
8,12,621
9,11,635


In [26]:
q("""
SELECT 
    bowler,
    COUNT(*) AS legal_balls,
    SUM(total_runs) AS runs_conceded,
    ROUND(6.0 * SUM(total_runs) / COUNT(*), 2) AS economy
FROM v_ball
WHERE is_wide_ball = 0 
  AND is_no_ball = 0
GROUP BY bowler
HAVING COUNT(*) >= 500
ORDER BY economy ASC
LIMIT 10;
""")

,bowler,legal_balls,runs_conceded,economy
0,A Kumble,965,1050,6.53
1,M Muralitharan,1524,1690,6.65
2,DW Steyn,2176,2438,6.72
3,SP Narine,4507,5103,6.79
4,DL Vettori,777,886,6.84
5,J Botha,694,799,6.91
6,SL Malinga,2827,3291,6.98
7,Harbhajan Singh,3416,3996,7.02
8,R Ashwin,4710,5521,7.03
9,R Sharma,928,1093,7.07


In [27]:
q("""
--who is the most economical bowler, counting only runs that were their fault?

SELECT bowler, SUM(is_legal) AS balls,
       ROUND(6.0*SUM(bowler_runs)/SUM(is_legal),2) AS b_economy

FROM v_ball
GROUP BY bowler
HAVING balls > 500
ORDER BY b_economy
LIMIT 10;
""")

,bowler,balls,b_economy
0,A Kumble,965,6.58
1,M Muralitharan,1524,6.68
2,DL Vettori,777,6.79
3,SP Narine,4507,6.79
4,DW Steyn,2176,6.92
5,J Botha,694,6.92
6,R Sharma,928,7.02
7,Harbhajan Singh,3416,7.08
8,SL Malinga,2827,7.14
9,Rashid Khan,3387,7.16


In [28]:
q("""
SELECT bowler, SUM(is_legal) AS total_balls, SUM(bowler_wicket) AS total_wickets,
        ROUND(1.0*SUM(is_legal)/SUM(bowler_wicket),2) AS balls_per_wicket

FROM v_ball
GROUP BY bowler
HAVING total_wickets > 50
ORDER BY balls_per_wicket;
""")

,bowler,total_balls,total_wickets,balls_per_wicket
0,AD Russell,1806,123,14.68
1,JR Hazlewood,1008,65,15.51
2,K Rabada,2112,133,15.88
3,MA Starc,1083,68,15.93
4,Imran Tahir,1316,82,16.05
...,...,...,...,...
73,I Sharma,2419,96,25.20
74,IK Pathan,2043,80,25.54
75,AR Patel,3483,136,25.61
76,JH Kallis,1742,65,26.80


In [29]:
q("""
SELECT wicket_kind, COUNT(*) AS n
FROM v_ball
WHERE is_wicket = 1
GROUP BY wicket_kind
ORDER BY n DESC;
""")

,wicket_kind,n
0,caught,9045
1,bowled,2398
2,run out,1167
3,lbw,871
4,caught and bowled,399
5,stumped,382
6,retired hurt,19
7,hit wicket,18
8,retired out,6
9,obstructing the field,4


In [30]:
q("""
SELECT
    CASE
        WHEN bowler_type LIKE '%Legbreak%'
          OR bowler_type LIKE '%Offbreak%'
          OR bowler_type LIKE '%Orthodox%'
          OR bowler_type LIKE '%Wrist spin%'
          OR bowler_type LIKE '%Googly%'
        THEN 'Spin'
        ELSE 'Pace'
    END AS style,
    SUM(is_legal) AS balls,
    ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2) AS economy,
    ROUND(100.0 * SUM(bowler_wicket) / SUM(is_legal), 2) AS wicket_pct
FROM v_ball
WHERE bowler_type IS NOT NULL
  AND TRIM(bowler_type) <> ''
GROUP BY style;
""")                                       

,style,balls,economy,wicket_pct
0,Pace,176859,8.57,4.93
1,Spin,100129,7.76,4.37


In [31]:
q("""SELECT CASE WHEN bowler_type LIKE 'Left%'
        THEN 'Left arm' ELSE 'Right arm' END
     AS arm,
     batsman_type,
     ROUND(6.0*SUM(bowler_runs)/SUM(is_legal),2)     
        AS economy,
     ROUND(100.0*SUM(bowler_wicket)/SUM(is_legal),2) 
        AS wicket_pct
  FROM  v_ball                                      
  WHERE TRIM(COALESCE(bowler_type,'')) <> ''         
    AND batsman_type IS NOT NULL                     
  GROUP BY arm, batsman_type; """)

,arm,batsman_type,economy,wicket_pct
0,Left arm,Left hand Bat,8.81,5.13
1,Left arm,Right hand Bat,8.49,4.97
2,Right arm,Left hand Bat,8.33,4.46
3,Right arm,Right hand Bat,8.16,4.78


In [32]:
q("""
 --which spinner is the best wicket-taker?
 --bowler,wicket,wicket_perc

SELECT
    bowler,
    SUM(bowler_wicket) AS wicket,
    ROUND(
        100.0 * SUM(bowler_wicket) / SUM(is_legal),
        2
    ) AS wicket_perc
FROM v_ball
WHERE bowler_type LIKE '%Legbreak%'
   OR bowler_type LIKE '%Offbreak%'
   OR bowler_type LIKE '%Orthodox%'
   OR bowler_type LIKE '%Wrist spin%'
   OR bowler_type LIKE '%Googly%'
GROUP BY bowler
HAVING SUM(is_legal) >= 1000
ORDER BY wicket DESC
LIMIT 10;
""")


,bowler,wicket,wicket_perc
0,YS Chahal,228,5.79
1,SP Narine,199,4.42
2,PP Chawla,192,4.99
3,R Ashwin,187,3.97
4,RA Jadeja,177,4.23
5,A Mishra,174,5.16
6,Rashid Khan,168,4.96
7,Harbhajan Singh,150,4.39
8,AR Patel,136,3.90
9,Kuldeep Yadav,109,4.90


In [33]:
q("""

--we need a death-over bowler.who is cheapest in over 16 to 20?
--bowler,balls,bowler_runs,economy

SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(bowler_runs) AS bowler_runs,
    ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2) AS economy
FROM v_ball
WHERE over_number BETWEEN 15 AND 19
GROUP BY bowler
HAVING SUM(is_legal) >=300
ORDER BY economy ASC
LIMIT 10;

""")

,bowler,balls,bowler_runs,economy
0,SP Narine,1075,1307,7.29
1,SL Malinga,1117,1464,7.86
2,JJ Bumrah,1393,1911,8.23
3,R Ashwin,601,825,8.24
4,DW Steyn,634,877,8.30
5,CV Varun,310,432,8.36
6,Rashid Khan,594,829,8.37
7,AR Patel,376,539,8.60
8,PP Chawla,444,641,8.66
9,CH Morris,646,933,8.67


In [34]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(bowler_wicket) AS wickets,
    ROUND(100.0 * SUM(bowler_wicket) / SUM(is_legal), 2) AS wicket_perc
    FROM v_ball
    WHERE over_number BETWEEN 0 AND 5
    GROUP BY bowler
    HAVING balls > 300
    ORDER BY wickets DESC
    LIMIT 10;
    """)

,bowler,balls,wickets,wicket_perc
0,B Kumar,2453,89,3.63
1,TA Boult,1644,72,4.38
2,DL Chahar,1527,67,4.39
3,Sandeep Sharma,1728,64,3.70
4,UT Yadav,1446,58,4.01
5,I Sharma,1521,57,3.75
6,Mohammed Shami,1584,57,3.60
7,Z Khan,1292,52,4.02
8,Mohammed Siraj,1326,50,3.77
9,R Ashwin,1252,50,3.99


In [35]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(is_legal AND total_runs = 0) AS dot_balls,
    ROUND(100.0 * SUM(is_legal AND total_runs = 0) / SUM(is_legal), 2) AS dot_perc
FROM v_ball
WHERE phase = 'Death'
GROUP BY bowler
HAVING balls >= 300
ORDER BY dot_perc DESC
LIMIT 10;
""")

,bowler,balls,dot_balls,dot_perc
0,SP Narine,1075,397,36.93
1,CV Varun,310,112,36.13
2,MG Johnson,302,108,35.76
3,Rashid Khan,594,208,35.02
4,DW Steyn,634,218,34.38
5,PP Chawla,444,150,33.78
6,K Rabada,660,222,33.64
7,M Prasidh Krishna,542,182,33.58
8,A Mishra,521,167,32.05
9,MA Starc,333,106,31.83


In [36]:
run_sql_file('../sql/08_matches_clean.sql')

ran ../sql/08_matches_clean.sql


In [37]:
q("""
--Does the toss winner win the match more often than not?
--decisive_matches, toss winner , per

SELECT 
    COUNT(*) AS decisive_matches,
    SUM(
        CASE 
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
    ) AS toss_winner_won,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN toss_winner = match_winner THEN 1
                ELSE 0
            END
            
        )/count(*),2) AS pct
FROM matches_clean
WHERE result = 'win';
""")

,decisive_matches,toss_winner_won,pct
0,1187,613,51.64


In [38]:
q("""
SELECT 
    toss_decision,
    COUNT(*) AS total_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM matches_clean), 1) AS perc
FROM 
    matches_clean
GROUP BY 
    toss_decision;
    """)

,toss_decision,total_count,perc
0,bat,412,34.0
1,field,800,66.0


In [39]:
q("""
SELECT 
    toss_winner, 
    COUNT(*) AS toss_wins
FROM matches
GROUP BY toss_winner
ORDER BY toss_wins DESC
LIMIT 10;
""")

,toss_winner,toss_wins
0,Mumbai Indians,156
1,Delhi Capitals,145
2,Sunrisers Hyderabad,139
3,Rajasthan Royals,134
4,Royal Challengers Bangalore,132
5,Kolkata Knight Riders,132
6,Chennai Super Kings,130
7,Punjab Kings,126
8,Gujarat Titans,33
9,Lucknow Super Giants,29


In [40]:
q("""
 --what is the average first-innings score?
 --first_innings_runs.v_matche_totals 


SELECT 
    AVG(first_innings_runs) AS avg_first_innings_score
FROM 
    v_match_totals
""")

,avg_first_innings_score
0,168.186184


In [41]:
q("""
SELECT 
    innings,
    ROUND(SUM(runs) * 6.0 / SUM(legal_balls), 2) AS run_rate
FROM v_innings
WHERE innings IN (1, 2)
GROUP BY innings
ORDER BY innings;
""")

,innings,run_rate
0,1,8.49
1,2,8.37


In [42]:
q("""

select * FROM v_innings;

""")

,match_id,innings,batting_team,bowling_team,runs,wickets,legal_balls
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,222,3,120
1,335982,2,Royal Challengers Bangalore,Kolkata Knight Riders,82,10,91
2,335983,1,Chennai Super Kings,Punjab Kings,240,5,120
3,335983,2,Punjab Kings,Chennai Super Kings,207,4,120
4,335984,1,Rajasthan Royals,Delhi Capitals,129,8,120
...,...,...,...,...,...,...,...
2413,1529284,2,Sunrisers Hyderabad,Mumbai Indians,249,4,112
2414,1529285,1,Royal Challengers Bangalore,Gujarat Titans,155,10,116
2415,1529285,2,Gujarat Titans,Royal Challengers Bangalore,158,6,95
2416,1529286,1,Rajasthan Royals,Delhi Capitals,225,6,120


In [43]:
q("""
--How often is a 200+ total successfully chased? 
--v_match_totals

SELECT COUNT(*) as match_with_200plus,
        sum(chase_won) as successfully_chased
FROM v_match_totals
WHERE first_innings_runs >= 200;
""")

,match_with_200plus,successfully_chased
0,198,46


In [44]:
q("""
--what is the biggest total ever defended , and the biggest ever chased?

SELECT MAX (CASE WHEN chase_won = 0 THEN first_innings_runs END)
        AS biggest_defend,
       MAX (CASE WHEN chase_won = 0 THEN first_innings_runs END)
        AS biggest_chase

FROM v_match_totals;




""")

,biggest_defend,biggest_chase
0,287,287


In [45]:
q("""
--Which ground has hosted the most matches?
--matches_clean table venue_clean column

SELECT
    venue_clean,
    COUNT(*) AS matches
FROM matches_clean
GROUP BY venue_clean
ORDER BY matches DESC
LIMIT 10;

""")

,venue_clean,matches
0,Wankhede Stadium,130
1,Eden Gardens,104
2,MA Chidambaram Stadium,95
3,M Chinnaswamy Stadium,89
4,Rajiv Gandhi International Stadium,87
5,Sawai Mansingh Stadium,66
6,Feroz Shah Kotla,60
7,Dubai International Cricket Stadium,46
8,Arun Jaitley Stadium,41
9,Narendra Modi Stadium,37


In [46]:
q("""
--which ground is the highest scoring?
SELECT m.venue_clean,
     count(DISTINCT m.match_id)as matches,\
     round(avg(i.runs),1) as avg_score
from matches_clean m
join v_innings i ON i.match_id=m.match_id

GROUP BY m.venue_clean
having matches>=25
ORDER BY avg_score DESC
LIMIT 10;


""")

,venue_clean,matches,avg_score
0,Arun Jaitley Stadium,41,180.0
1,Narendra Modi Stadium,37,177.8
2,Brabourne Stadium,27,172.3
3,Punjab Cricket Association IS Bindra Stadium,26,170.7
4,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,26,167.6
5,Wankhede Stadium,130,166.9
6,M Chinnaswamy Stadium,89,164.8
7,Sawai Mansingh Stadium,66,161.1
8,Rajiv Gandhi International Stadium,87,161.1
9,Eden Gardens,104,160.9


In [47]:
q("""
select match_winner, count(*)as wins
from matches_clean
where result='win'
group by match_winner
order by wins desc;
""")

,match_winner,wins
0,Mumbai Indians,153
1,Chennai Super Kings,145
2,Royal Challengers Bangalore,138
3,Kolkata Knight Riders,136
4,Sunrisers Hyderabad,128
5,Punjab Kings,125
6,Delhi Capitals,122
7,Rajasthan Royals,120
8,Gujarat Titans,42
9,Lucknow Super Giants,32


In [48]:
q("""
SELECT batting_team,
    ROUND(6.0 * SUM(total_runs) / SUM(is_legal), 2) AS run_rate
FROM v_ball
WHERE phase = 'Powerplay'
GROUP BY batting_team
HAVING SUM(is_legal) > 3000
ORDER BY run_rate DESC;
""")

,batting_team,run_rate
0,Punjab Kings,8.21
1,Sunrisers Hyderabad,8.20
2,Rajasthan Royals,8.07
3,Delhi Capitals,8.03
4,Kolkata Knight Riders,8.02
5,Mumbai Indians,7.93
6,Royal Challengers Bangalore,7.91
7,Chennai Super Kings,7.85


In [49]:
q("""
 --which team relies most on boundaries for its runs?
 --run rate 100*batter_runs/legal_ball 4 6 only having sum(is_legal)>=10000
SELECT
    batting_team AS team,
    SUM(CASE WHEN batter_runs IN (4, 6) THEN batter_runs ELSE 0 END) AS boundary_runs,
    SUM(total_runs) AS total_runs,
    ROUND(
        100.0 * SUM(CASE WHEN batter_runs IN (4, 6) THEN batter_runs ELSE 0 END)
        / SUM(batter_runs), 2
    ) AS boundary_percentage
FROM v_ball
GROUP BY batting_team
HAVING SUM(is_legal) >=10000
ORDER BY boundary_percentage DESC
LIMIT 10;
""")

,team,boundary_runs,total_runs,boundary_percentage
0,Mumbai Indians,27486,46517,62.30
1,Kolkata Knight Riders,24846,42540,61.84
2,Punjab Kings,25986,44352,61.67
3,Royal Challengers Bangalore,26220,44890,61.44
4,Rajasthan Royals,22718,39159,60.90
5,Delhi Capitals,24688,43297,59.97
6,Chennai Super Kings,24186,42557,59.80
7,Sunrisers Hyderabad,24982,44948,58.50


In [50]:
q("""
-- which player has won the most man of the match awards?
--player,award_count
--matches_clean.player_of_match?

SELECT
    player_of_match AS player,
    COUNT(*) AS award_count
FROM matches_clean
GROUP BY player_of_match
ORDER BY award_count DESC
LIMIT 10;
""")

,player,award_count
0,AB de Villiers,25
1,CH Gayle,22
2,RG Sharma,21
3,V Kohli,20
4,MS Dhoni,18
5,DA Warner,18
6,SP Narine,17
7,RA Jadeja,17
8,KL Rahul,17
9,YK Pathan,16


In [51]:
q("""
 --How does the way batters  get out change across the phases?
 --Phases,wicket_kind = caught,bowled,run out, 
 --v_ball table is_wicket
 
 select phase,wicket_kind, count(*) as n
 from v_ball
 where is_wicket=1 and wicket_kind in ('caught','bowled','run out')
 group by phase,wicket_kind
 order by  phase,n desc;
""")

,phase,wicket_kind,n
0,Death,caught,3289
1,Death,bowled,877
2,Death,run out,634
3,Middle,caught,3484
4,Middle,bowled,909
5,Middle,run out,344
6,Powerplay,caught,2272
7,Powerplay,bowled,612
8,Powerplay,run out,189


In [52]:
q("""
 --which batter is most often run out?
 --Player_name,run_out_count
 --player_out , wicket_kind = 'run out'
 
 select player_out,count(*) as run_out
 from v_ball
 where wicket_kind='run out'
 group by player_out
 order by run_out desc
 limit  10;
""")

,player_out,run_out
0,S Dhawan,16
1,G Gambhir,16
2,SK Raina,15
3,KD Karthik,15
4,AT Rayudu,15
5,AB de Villiers,14
6,M Vijay,12
7,F du Plessis,12
8,YK Pathan,11
9,Y Venugopal Rao,11


In [54]:
q("""
 --how many matches were played each season
 select season_year , count(*) as n
from matches_clean
group by  season_year
order by season_year;
""")

,season_year,n
0,2008,58
1,2009,57
2,2010,60
3,2011,73
4,2012,74
5,2013,76
6,2014,60
7,2015,59
8,2016,60
9,2017,59


In [ ]:
q("""
 --Which bowler dismissef a named batter most often
 
""")

In [61]:
q("""
--which bowler dismissed a named batter most often ?
-- V Kohli
--v_ball
--player_out='V kohli' and bowler_wicket=1?

SELECT bowler,  COUNT(*) as n
FROM  v_ball
WHERE  player_out = 'V Kohli' AND bowler_wicket = 1
GROUP BY  bowler
order by n desc
limit 5;   
""")

,bowler,n
0,Sandeep Sharma,7
1,A Nehra,6
2,JJ Bumrah,5
3,Mohammed Shami,5
4,Avesh Khan,4


In [68]:
q("""
 --Find an opener who scores fast and survives the powerplay.
 --batter,balls,runs,sr
 --havinng balls >=500
 SELECT
    batter,
    SUM(is_legal) AS balls,
    SUM(batter_runs) AS runs,
    ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS strike_rate
FROM v_ball
WHERE phase = 'Powerplay' and is_wicket=0
GROUP BY batter
HAVING balls >= 500
ORDER BY strike_rate DESC
LIMIT 10;
""")

,batter,balls,runs,strike_rate
0,SP Narine,640,1174,183.44
1,Abhishek Sharma,712,1287,180.76
2,YBK Jaiswal,948,1579,166.56
3,P Simran Singh,658,1070,162.61
4,JM Bairstow,601,944,157.07
5,PP Shaw,866,1347,155.54
6,V Sehwag,1036,1593,153.76
7,CA Lynn,520,779,149.81
8,RA Tripathi,685,1013,147.88
9,JC Buttler,1216,1788,147.04


In [69]:
q("""
select b.batter as players,sum(b.batter_runs)as runs,(select sum (x.bowler_wicket)from v_ball x where x.bowler=b.batter)as wickets
from v_ball b
group by b.batter
having runs>=1000 and wickets >=50
order by runs desc
limit 10;
""")

,players,runs,wickets
0,SR Watson,3874,92
1,KA Pollard,3412,69
2,RA Jadeja,3412,177
3,HH Pandya,2877,82
4,AD Russell,2651,123
5,JH Kallis,2427,65
6,AR Patel,1947,136
7,SP Narine,1820,199
8,KH Pandya,1796,102
9,DJ Bravo,1560,183


In [73]:
q("""
SELECT
    batter,
    SUM(is_legal) AS balls,
    ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS sr
FROM v_ball
WHERE innings = 2
  AND phase = 'Death'
GROUP BY batter
HAVING SUM(is_legal) >= 200
ORDER BY sr DESC
LIMIT 10;
""")


,batter,balls,sr
0,AB de Villiers,295,209.83
1,SS Iyer,215,187.44
2,DA Miller,439,186.33
3,RR Pant,214,185.51
4,SV Samson,207,183.57
5,KA Pollard,429,183.45
6,SO Hetmyer,217,180.65
7,V Kohli,336,180.65
8,AD Russell,265,174.72
9,YK Pathan,331,174.32
